In [ ]:
import pandas as pd

In [ ]:
feature_df = pd.read_csv("/content/drive/MyDrive/Disease_engine_project/processed_data/features.csv")

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Disease_engine_project/release_validate_patients_pipe_split", sep='|')
## The path should be the same path of the data that will be processed

In [ ]:
data = data[["AGE", "PATHOLOGY", "EVIDENCES"]]

In [ ]:
data["EVIDENCES"] = data["EVIDENCES"].str.replace(r'(\'| |\[|\]|@_)', '', regex=True).str.split(',')
# Transforms the string formats into the same as the features in the feature dataset, casts the string into an array.
# Uses vectorized functions

In [ ]:
ENCODING = {
    "yes":1,
    "no":-1,
    "unanswered":0
}

In [ ]:
new_df = feature_df.iloc[:,:]
new_rows = [new_df]
for i in range(0, data.shape[0]): #Last checkpoint starts at part 700001, to start from scratch, set the range to start at 0}

  data_row = data.iloc[i, :] ## get the i_th medical record
  evidences = data_row[2] ## get the array of evidences preprocessed in the previous step
  disease = [data_row[1]] ## get the name of the disease
  age = data_row[0] ## get the age of the individual from the given record
  row = {}
  row["Age"] = [int(age)]
  row["Diagnosis"] = disease

  for j in evidences:
    row[j] = [ENCODING["yes"]] ## Sets the present evidences to 1 to signal the present evidences for the given record
  evidences = []
  new_rows.append(pd.DataFrame(row)) ## Constructs row from the dictionary of present evidences

  if i % 10000 == 0 and i != 0:

    ## Every 10k rows processed, merge all the rows that have been processed, concatenate them with the
    ## dataframe containing all the possible fratures, replace NaN values for 0, and store in a parquet

    concat_df = pd.concat(new_rows, ignore_index=True).fillna(ENCODING["unanswered"])
    new_rows = [new_df]
    concat_df.to_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_" + str(i) + ".parquet", engine='pyarrow')

In [ ]:
for i in range(10000, data.shape[0], 10000):
  vectors = pd.read_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_" + str(i) +".parquet", engine='pyarrow')
  vectors.replace(ENCODING["unanswered"], ENCODING["no"])
  concat_df.to_parquet("/content/drive/MyDrive/Disease_engine_new_encoding/processed_data/validate_" + str(i) + ".parquet", engine='pyarrow')

## For testing and validating, this cell generates the dataset with the second type of encoding featuring 1 representing yes and -1 representing no

Two representations have been created. The first one represents 1 as Yes while 0 represents No. The second one represents 1 as Yes, -1 as No, and 0 as unanswered. Both representations will be tested